# Superstore Orders - Exploratory Data Analysis
**Project:** Retail Data Warehouse  
**Author:** Jovan Paic  
**Purpose:** Understand the raw dataset before ETL — shape, types, nulls, and key distributions that informed pipeline design decisions.

---

## 1 — Imports & Config

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titlesize']    = 13
plt.rcParams['axes.labelsize']    = 11

## 2 — Load Raw Data

In [ ]:
df = pd.read_csv('../data/raw/SuperStoreOrders - SuperStoreOrders.csv')

print(f'Rows : {df.shape[0]:,}')
print(f'Cols : {df.shape[1]}')
df.head()

## 3 — Schema & Data Types
All fields arrive as raw strings from the CSV.  
This confirms why the first SQL step (`03_raw_to_staging.sql`) casts every column
to the correct type — dates to `DATE`, numerics to `NUMERIC` / `INTEGER`.

In [ ]:
df.dtypes.to_frame('dtype')

## 4 — Missing Values
Identify columns with nulls before ingestion.  
Any nulls in `order_id`, `customer_name`, or `sales` would block loading
into the staging layer — these are checked in `05_missing_values.sql`.

In [ ]:
null_summary = df.isnull().sum()
null_summary = null_summary[null_summary > 0]

if null_summary.empty:
    print('No nulls found in raw data.')
else:
    print(null_summary)

## 5 — Duplicate Detection
One `order_id` can legitimately have multiple lines (different products).  
True duplicates are identical on `order_id + product_id + order_date + sales + quantity`.  
These are removed in `04_duplicate_removal.sql`.

In [ ]:
total_rows  = len(df)
exact_dupes = df.duplicated(subset=['order_id','product_id','order_date','sales','quantity']).sum()

print(f'Total rows      : {total_rows:,}')
print(f'True duplicates : {exact_dupes:,}')
print(f'Clean rows      : {total_rows - exact_dupes:,}')

## 6 — Numeric Formatting Issues
`sales`, `profit`, and `shipping_cost` sometimes contain commas (e.g. `1,234.56`).  
This is why `03_raw_to_staging.sql` uses `REPLACE(sales, ',', '')` before casting.

In [ ]:
for col in ['sales', 'profit', 'shipping_cost']:
    has_comma = df[col].astype(str).str.contains(',').sum()
    print(f'{col:>15} — rows with commas: {has_comma:,}')

## 7 — Revenue by Category

In [ ]:
df['sales_clean'] = pd.to_numeric(df['sales'].astype(str).str.replace(',',''), errors='coerce')

cat_rev = (
    df.groupby('category')['sales_clean']
      .sum()
      .sort_values(ascending=True)
)

fig, ax = plt.subplots()
bars = ax.barh(cat_rev.index, cat_rev.values, color=['#4C72B0','#DD8452','#55A868'])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Total Revenue by Category')
ax.set_xlabel('Revenue (USD)')
ax.bar_label(bars, labels=[f'${v:,.0f}' for v in cat_rev.values], padding=5)
plt.tight_layout()
plt.show()

## 8 — Order Volume by Year

In [ ]:
orders_by_year = df.groupby('year').size()

fig, ax = plt.subplots()
ax.bar(orders_by_year.index, orders_by_year.values, color='#4C72B0', width=0.5)
ax.set_title('Number of Order Lines by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Order Lines')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, (yr, val) in enumerate(orders_by_year.items()):
    ax.text(yr, val + 100, f'{val:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 9 — Revenue by Market

In [ ]:
market_rev = (
    df.groupby('market')['sales_clean']
      .sum()
      .sort_values(ascending=False)
)

fig, ax = plt.subplots()
ax.bar(market_rev.index, market_rev.values, color='#4C72B0')
ax.set_title('Total Revenue by Market')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 10 — Revenue by Customer Segment

In [ ]:
seg_rev = df.groupby('segment')['sales_clean'].sum()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    seg_rev.values,
    labels=seg_rev.index,
    autopct='%1.1f%%',
    colors=['#4C72B0','#DD8452','#55A868'],
    startangle=140
)
ax.set_title('Revenue Share by Customer Segment')
plt.tight_layout()
plt.show()

## 11 — Profit Distribution
Negative profit values exist — these are valid loss-making orders, not data errors.

In [ ]:
df['profit_clean'] = pd.to_numeric(df['profit'].astype(str).str.replace(',',''), errors='coerce')

fig, ax = plt.subplots()
ax.hist(df['profit_clean'].dropna(), bins=80, color='#4C72B0', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Break-even')
ax.set_title('Profit Distribution per Order Line')
ax.set_xlabel('Profit (USD)')
ax.set_ylabel('Frequency')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

loss_rows = (df['profit_clean'] < 0).sum()
print(f'Loss-making order lines: {loss_rows:,} ({loss_rows/len(df)*100:.1f}% of total)')

## 12 — Key Findings & Pipeline Decisions

| Finding | Pipeline Decision |
|---|---|
| All fields arrive as TEXT | Cast in `03_raw_to_staging.sql` |
| Commas in numeric fields | `REPLACE(col, ',', '')` before `CAST` |
| Same `order_id` → multiple products | Duplicate check on 5-column key, not just `order_id` |
| Same `product_id` → varying names | `GROUP BY product_id` with `MIN()` in dimension load |
| No city field in source | `city = NULL` in `dim_geography` |
| Negative profit values exist | Kept as-is — valid business data |

---